# Local AI Sales Intelligence

A complete source walkthrough in ten chapters. Each chapter contains explanations, full file-writing code cells, runnable fictional-data examples and a verification checkpoint.

Start Jupyter from this project using its virtual environment. Read `README.md` first. The delivered source files are already present. Source cells reconstruct a separate copy in `notebook_project/`; demonstrations import the supplied, tested package.

All demonstrations use clearly fictional transactions. Model and n8n calls are opt-in. `Run All` does not install system software, download models, publish workflows, or start services. To review one chapter at a time, use the individual notebooks and pause at each checkpoint.

The original prompt's ten topics are retained. Basic deterministic preparation comes before optional local models, and the complete web API is assembled after its modules exist.


In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the supplied project or its notebooks directory."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Source-writing cells reconstruct a separate copy and preserve the supplied project.
BUILD_ROOT = ROOT / "notebook_project"
BUILD_ROOT.mkdir(exist_ok=True)
for folder in ["sales_intelligence", "scripts", "tests", "workflows", "frontend/src", "docs"]:
    (BUILD_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_LOCAL_MODELS = False  # opt in only after installing local Ollama models
RUN_N8N = False          # opt in only after configuring/publishing the local workflows
print("Python:", sys.version.split()[0])
print("Source reconstruction:", BUILD_ROOT)


## Chapter 10 · Simulation, reporting, deployment and verification

Calculate explicit arithmetic scenarios, export safe CSV/XLSX/JSON/HTML, document assumptions, package local deployment and run the full acceptance tests. Cleanup previews expire-after-inactivity candidates before explicit deletion.

### Code cell · `sales_intelligence/reporting.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'sales_intelligence/reporting.py'}"
import html
import io
import json

import pandas as pd

from .analytics import analyze
from .preparation import load_clean
from .schemas import Simulation


def simulate(store, dataset_id, request: Simulation):
    if not request.assumptions_confirmed:
        raise ValueError(
            "Confirm that this is arithmetic under assumptions, not an estimated demand response."
        )
    frame, data = load_clean(store, dataset_id)
    if frame._net_sales.isna().any() or (frame._net_sales < 0).any():
        raise ValueError("Simulation requires complete, nonnegative net amounts; analyze returns separately.")
    new_discount = request.current_discount_rate + request.discount_change_points
    if not 0 <= new_discount < 1:
        raise ValueError("The resulting discount rate must be between zero and one, excluding one.")
    baseline = float(frame._net_sales.sum())
    gross = baseline / (1 - request.current_discount_rate)
    hypothetical = gross * (1 + request.price_change) * (1 + request.volume_change) * (1 - new_discount)
    result = {
        "dataset_version": data["version"],
        "currency": data["meta"]["mapping"]["currency"],
        "baseline_net_sales": baseline,
        "hypothetical_net_sales": hypothetical,
        "difference": hypothetical - baseline,
        "assumptions": request.model_dump(),
        "formula": "baseline_net/(1-current_discount) * (1+price_change) * (1+volume_change) * (1-new_discount)",
        "kind": "arithmetic_scenario",
        "causal_prediction": False,
        "limitations": [
            "Applies one user-confirmed average discount rate to all sales.",
            "Volume response is supplied by the user, not learned from observational data.",
            "Retention and CLV changes need a separately validated behavioral model.",
        ],
    }
    if "_total_cost" in frame and frame._total_cost.notna().all():
        result["hypothetical_profit"] = hypothetical - float(frame._total_cost.sum()) * (
            1 + request.volume_change
        )
        result["cost_assumption"] = (
            "Costs scale linearly with volume; unit costs stay fixed. Fixed costs are not separated."
        )
    return result


def safe_cell(value):
    if isinstance(value, str) and value.lstrip().startswith(("=", "+", "-", "@", "\t", "\r")):
        return "'" + value
    return value


def export(store, dataset_id, format):
    frame, data = load_clean(store, dataset_id)
    safe = frame.map(safe_cell)
    if format == "csv":
        return safe.to_csv(index=False).encode("utf-8-sig"), "text/csv; charset=utf-8", "cleaned.csv"
    if format == "xlsx":
        output = io.BytesIO()
        with pd.ExcelWriter(output, engine="openpyxl") as writer:
            safe.to_excel(writer, sheet_name="Cleaned", index=False)
            pd.DataFrame([{"metadata": json.dumps(data["meta"], ensure_ascii=False)}]).to_excel(
                writer, sheet_name="Metadata", index=False
            )
        return (
            output.getvalue(),
            "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "report.xlsx",
        )
    report = {"summary": analyze(store, dataset_id), "quality": data["meta"]["quality"]}
    if format == "json":
        return (
            json.dumps(report, ensure_ascii=False, allow_nan=False, indent=2).encode(),
            "application/json",
            "report.json",
        )
    if format == "html":
        text = html.escape(json.dumps(report, ensure_ascii=False, indent=2))
        body = f'<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8"><title>گزارش فروش</title><body><h1>گزارش فروش</h1><pre dir="auto">{text}</pre></body></html>'
        return body.encode(), "text/html; charset=utf-8", "report.html"
    raise ValueError("Supported exports: csv, xlsx, json, html.")


### Code cell · `scripts/cleanup.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/cleanup.py'}"
"""Preview or explicitly delete expired datasets. Original files are included only on explicit deletion."""

import argparse
import shutil
import sqlite3
from datetime import UTC, datetime, timedelta

from sales_intelligence.config import Settings
from sales_intelligence.store import Store

parser = argparse.ArgumentParser()
parser.add_argument(
    "--apply", action="store_true", help="Actually delete expired datasets, original files and history"
)
arguments = parser.parse_args()
settings = Settings().prepare()
store = Store(settings.data_dir)
cutoff = (datetime.now(UTC) - timedelta(days=settings.retention_days)).isoformat()
with store.db() as db:
    candidates = db.execute(
        "SELECT id FROM datasets WHERE updated < ? AND id NOT IN (SELECT dataset_id FROM tasks WHERE status IN ('queued','running'))",
        (cutoff,),
    ).fetchall()
    for row in candidates:
        dataset_id = row["id"]
        print(dataset_id, "delete" if arguments.apply else "would expire")
        if arguments.apply:
            checkpoints = store.root / "graph_state.sqlite"
            if checkpoints.exists():
                with sqlite3.connect(checkpoints) as graph_db:
                    for table in ["checkpoints", "writes"]:
                        exists = graph_db.execute(
                            "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (table,)
                        ).fetchone()
                        if exists:
                            graph_db.execute(
                                f"DELETE FROM {table} WHERE thread_id LIKE ?", (dataset_id + ":%",)
                            )
            shutil.rmtree(store.directory(dataset_id))
            db.execute("DELETE FROM audit WHERE dataset_id=?", (dataset_id,))
            db.execute("DELETE FROM tasks WHERE dataset_id=?", (dataset_id,))
            db.execute("DELETE FROM datasets WHERE id=?", (dataset_id,))
print(
    "Preview only. Add --apply to delete expired datasets." if not arguments.apply else "Cleanup completed."
)


### Code cell · `scripts/notebook_demo.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/notebook_demo.py'}"
"""Reusable fictional-data setup so every chapter can run independently."""

import tempfile
from pathlib import Path

from sales_intelligence.config import Settings
from sales_intelligence.ingestion import ingest
from sales_intelligence.preparation import clean, save_mapping
from sales_intelligence.schemas import Cleaning, Mapping
from sales_intelligence.store import Store
from scripts.make_sample import create_sample


def make_demo(root):
    directory = Path(root) / "notebook_data"
    directory.mkdir(exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix="fictional-", dir=directory))
    settings = Settings(
        api_key="notebook-fictional-only-" + "x" * 32, data_dir=run_dir / "data", _env_file=None
    ).prepare()
    store = Store(settings.data_dir)
    frame = create_sample(run_dir / "sample")
    data = ingest(store, settings, "sales.csv", frame.to_csv(index=False).encode())
    columns = {
        name: name
        for name in [
            "order_id",
            "customer_id",
            "product_id",
            "transaction_date",
            "sales_quantity",
            "sales_amount",
            "cost",
            "discount",
        ]
    }
    columns["text"] = "feedback"
    save_mapping(
        store,
        data["id"],
        Mapping(
            sheet="CSV",
            columns=columns,
            grain="line_item",
            currency="DEMO",
            sales_definition="Fictional net line amounts; no tax, shipping or returns; demonstration units.",
            cost_basis="line_total",
            confirmed=True,
        ),
    )
    clean(store, data["id"], Cleaning(confirmed=True))
    return store, settings, data["id"], frame


### Code cell · `Dockerfile`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'Dockerfile'}"
FROM python:3.13-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY pyproject.toml README.md requirements.lock.txt ./
COPY sales_intelligence ./sales_intelligence
RUN pip install --no-cache-dir -r requirements.lock.txt && pip install --no-cache-dir --no-deps .
COPY scripts ./scripts
RUN useradd --create-home --uid 10001 sales && mkdir /data && chown sales:sales /data
USER sales
EXPOSE 8000
CMD ["python", "scripts/serve.py"]


### Code cell · `.dockerignore`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / '.dockerignore'}"
.git
.env
.venv
data
notebook_data
frontend
tests
notebooks
sample_data
.cache


### Code cell · `compose.yaml`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'compose.yaml'}"
services:
  api:
    build: .
    environment:
      SALES_API_KEY: ${SALES_API_KEY:?Run scripts/bootstrap.py first}
      SALES_DATA_DIR: /data
      SALES_OLLAMA_URL: http://ollama:11434
      SALES_LLM_MODEL: ${SALES_LLM_MODEL:-qwen3:8b}
      SALES_EMBEDDING_MODEL: ${SALES_EMBEDDING_MODEL:-nomic-embed-text}
      SALES_USE_LLM: ${SALES_USE_LLM:-false}
      SALES_BIND: 0.0.0.0
      LANGSMITH_TRACING: 'false'
      LANGCHAIN_TRACING_V2: 'false'
    ports:
      - '127.0.0.1:8000:8000'
    volumes:
      - sales_data:/data
    restart: unless-stopped
  ollama:
    image: ${OLLAMA_IMAGE:-ollama/ollama:0.40.14}
    environment:
      OLLAMA_NO_CLOUD: '1'
      OLLAMA_NUM_PARALLEL: '1'
      OLLAMA_MAX_LOADED_MODELS: '1'
    volumes:
      - ollama_data:/root/.ollama
    restart: unless-stopped
  n8n:
    image: ${N8N_IMAGE:-docker.n8n.io/n8nio/n8n:2.46.4}
    ports:
      - '127.0.0.1:5678:5678'
    environment:
      N8N_ENCRYPTION_KEY: ${N8N_ENCRYPTION_KEY:?Run scripts/bootstrap.py first}
      N8N_ENFORCE_SETTINGS_FILE_PERMISSIONS: 'true'
      N8N_SECURE_COOKIE: 'false'
      N8N_DIAGNOSTICS_ENABLED: 'false'
      N8N_VERSION_NOTIFICATIONS_ENABLED: 'false'
      N8N_TEMPLATES_ENABLED: 'false'
      N8N_PERSONALIZATION_ENABLED: 'false'
      EXECUTIONS_DATA_SAVE_ON_SUCCESS: none
      EXECUTIONS_DATA_SAVE_ON_ERROR: none
      EXECUTIONS_DATA_SAVE_MANUAL_EXECUTIONS: 'false'
      EXECUTIONS_DATA_PRUNE: 'true'
      EXECUTIONS_DATA_MAX_AGE: '24'
      GENERIC_TIMEZONE: Asia/Tehran
      TZ: Asia/Tehran
    volumes:
      - n8n_data:/home/node/.n8n
      - ./workflows:/workflows:ro
    restart: unless-stopped
  web:
    build: ./frontend
    ports:
      - '127.0.0.1:5173:80'
    depends_on:
      - api
    restart: unless-stopped
volumes:
  sales_data:
  ollama_data:
  n8n_data:


### Code cell · `frontend/Dockerfile`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'frontend/Dockerfile'}"
FROM node:24-alpine AS build
WORKDIR /web
COPY package*.json ./
RUN npm ci
COPY . .
RUN npm run build
FROM nginx:alpine
COPY nginx.conf /etc/nginx/conf.d/default.conf
COPY --from=build /web/dist /usr/share/nginx/html
EXPOSE 80


### Code cell · `frontend/nginx.conf`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'frontend/nginx.conf'}"
server {
    listen 80;
    server_name localhost;
    root /usr/share/nginx/html;
    client_max_body_size 26m;
    location /api/ {
        proxy_pass http://api:8000;
        proxy_read_timeout 180s;
        proxy_set_header Host $host;
    }
    location / { try_files $uri $uri/ /index.html; }
}


### Code cell · `tests/conftest.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'tests/conftest.py'}"
import pytest

from sales_intelligence.config import Settings
from sales_intelligence.ingestion import ingest
from sales_intelligence.preparation import clean, save_mapping
from sales_intelligence.schemas import Cleaning, Mapping
from sales_intelligence.store import Store


@pytest.fixture
def settings(tmp_path):
    return Settings(api_key="test-only-" + "a" * 32, data_dir=tmp_path, _env_file=None).prepare()


@pytest.fixture
def store(settings):
    return Store(settings.data_dir)


@pytest.fixture
def prepared(store, settings):
    body = b"order_id,customer_id,product_id,transaction_date,sales_quantity,sales_amount,cost\nO1,C1,P1,2025-01-01,2,100,60\nO1,C1,P2,2025-01-01,1,50,30\nO2,C2,P1,2025-01-02,1,80,40\nO3,,P2,2025-01-03,1,20,10\n"
    dataset = ingest(store, settings, "sales.csv", body)
    dataset_id = dataset["id"]
    columns = [
        "order_id",
        "customer_id",
        "product_id",
        "transaction_date",
        "sales_quantity",
        "sales_amount",
        "cost",
    ]
    save_mapping(
        store,
        dataset_id,
        Mapping(
            sheet="CSV",
            columns={c: c for c in columns},
            grain="line_item",
            currency="DEMO",
            sales_definition="Net amounts after discounts, excluding tax and shipping.",
            cost_basis="line_total",
            confirmed=True,
        ),
    )
    clean(store, dataset_id, Cleaning(confirmed=True))
    return dataset_id


### Code cell · `tests/test_data.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'tests/test_data.py'}"
import io
import json
import zipfile

import openpyxl
import pandas as pd
import pytest

from sales_intelligence.analytics import analyze, dashboard
from sales_intelligence.ingestion import check_archive, ingest
from sales_intelligence.preparation import clean, load_clean, number, save_mapping, save_relationship
from sales_intelligence.reporting import export, simulate
from sales_intelligence.schemas import Cleaning, Filters, Mapping, Relationship, Simulation


def test_verified_totals_and_order_grain(store, prepared):
    result = analyze(store, prepared)["result"]
    assert result["net_sales"] == 250
    assert result["orders"] == 3
    assert result["average_order_value"] == pytest.approx(250 / 3)
    assert result["profit"] == 110
    assert analyze(store, prepared, "products")["result"]["ranking"][0]["net_sales"] == 180


def test_filters_change_real_query(store, prepared):
    result = analyze(store, prepared, filters=Filters(start="2025-01-02", end="2025-01-02"))
    assert result["result"]["net_sales"] == 80
    assert result["filters"]["start"] == "2025-01-02"
    with pytest.raises(ValueError, match="No rows"):
        analyze(store, prepared, filters=Filters(customer_id="UNKNOWN"))


def test_null_ids_preserved_and_quality_reported(store, prepared):
    frame, data = load_clean(store, prepared)
    assert pd.isna(frame.iloc[-1].customer_id)
    assert data["meta"]["quality"]["nulls"]["customer_id"] == 1
    assert analyze(store, prepared, "customers")["result"]["excluded_rows"] == 1
    assert (store.directory(prepared) / "original.csv").exists()
    changes = list(store.directory(prepared).glob("*_changes.json"))
    assert changes and json.loads(changes[0].read_text())


def test_mapping_requires_human_confirmation():
    with pytest.raises(ValueError, match="person"):
        Mapping(
            sheet="CSV",
            columns={},
            grain="order",
            currency="IRT",
            sales_definition="Net sales after discount.",
        )


def test_purchase_amount_not_automatically_sales(store, settings):
    dataset = ingest(store, settings, "p.csv", b"date,purchase_amount\n2025-01-01,100\n")
    from sales_intelligence.preparation import inspect

    assert "sales_amount" not in inspect(store, dataset["id"])["sheets"]["CSV"]["mapping_suggestions"]


@pytest.mark.parametrize("text,expected", [("۱٬۲۳۴٫۵", 1234.5), ("۱۲۳", 123), ("-۱۲", -12)])
def test_persian_numbers(text, expected):
    assert number(text, Cleaning(thousands_separator="٬", confirmed=True)) == expected


def test_ambiguous_number_rejected():
    with pytest.raises(ValueError):
        number("12,34", Cleaning(confirmed=True))


def test_jalali_date_and_exact_duplicates(store, settings):
    dataset = ingest(
        store, settings, "p.csv", "date,amount,id\n۱۴۰۴/۰۱/۰۱,۱۰۰,001\n۱۴۰۴/۰۱/۰۱,۱۰۰,001\n".encode()
    )
    save_mapping(
        store,
        dataset["id"],
        Mapping(
            sheet="CSV",
            columns={"transaction_date": "date", "sales_amount": "amount", "customer_id": "id"},
            grain="line_item",
            currency="IRT",
            sales_definition="Net sales, tax and shipping excluded.",
            date_calendar="jalali",
            date_format="%Y/%m/%d",
            confirmed=True,
        ),
    )
    report = clean(store, dataset["id"], Cleaning(duplicates="drop_exact", confirmed=True))
    frame, _ = load_clean(store, dataset["id"])
    assert report["rows_before"] == 2 and report["rows_after"] == 1
    assert report["removed_source_rows"] == [3]
    assert frame.iloc[0].customer_id == "001"
    assert str(frame.iloc[0].transaction_date.date()) == "2025-03-21"


def test_duplicate_headers_are_not_silently_renamed(store, settings):
    with pytest.raises(ValueError, match="Duplicate column"):
        ingest(store, settings, "p.csv", b"date,date\n1,2\n")


def workbook_bytes(sales, dimensions):
    output = io.BytesIO()
    with pd.ExcelWriter(output, engine="openpyxl") as writer:
        sales.to_excel(writer, sheet_name="Sales", index=False)
        dimensions.to_excel(writer, sheet_name="Products", index=False)
    return output.getvalue()


def test_many_to_many_join_rejected(store, settings):
    body = workbook_bytes(
        pd.DataFrame({"date": ["2025-01-01"], "amount": [100], "sku": ["P1"]}),
        pd.DataFrame({"key": ["P1", "P1"], "category": ["A", "B"]}),
    )
    dataset = ingest(store, settings, "sales.xlsx", body)
    save_mapping(
        store,
        dataset["id"],
        Mapping(
            sheet="Sales",
            columns={"transaction_date": "date", "sales_amount": "amount", "product_id": "sku"},
            grain="line_item",
            currency="DEMO",
            sales_definition="Net after discounts and tax excluded.",
            confirmed=True,
        ),
    )
    with pytest.raises(ValueError, match="double counting"):
        save_relationship(
            store,
            dataset["id"],
            Relationship(
                sheet="Products", left_on="sku", right_on="key", columns=["category"], confirmed=True
            ),
        )


def test_join_preserves_sales_and_all_sheets(store, settings):
    body = workbook_bytes(
        pd.DataFrame({"date": ["2025-01-01", "2025-01-01"], "amount": [100, 50], "sku": ["P1", "P1"]}),
        pd.DataFrame({"key": ["P1"], "category": ["A"]}),
    )
    dataset = ingest(store, settings, "../../sales.xlsx", body)
    assert set(dataset["meta"]["sheets"]) == {"Sales", "Products"}
    save_mapping(
        store,
        dataset["id"],
        Mapping(
            sheet="Sales",
            columns={"transaction_date": "date", "sales_amount": "amount", "product_id": "sku"},
            grain="line_item",
            currency="DEMO",
            sales_definition="Net after discounts and tax excluded.",
            confirmed=True,
        ),
    )
    save_relationship(
        store,
        dataset["id"],
        Relationship(sheet="Products", left_on="sku", right_on="key", columns=["category"], confirmed=True),
    )
    clean(store, dataset["id"], Cleaning(confirmed=True))
    assert analyze(store, dataset["id"])["result"]["net_sales"] == 150


def test_workbook_formulas_rejected(store, settings):
    output = io.BytesIO()
    book = openpyxl.Workbook()
    sheet = book.active
    sheet.append(["date", "amount"])
    sheet.append(["2025-01-01", "=1+1"])
    book.save(output)
    with pytest.raises(ValueError, match="formulas"):
        ingest(store, settings, "bad.xlsx", output.getvalue())


@pytest.mark.parametrize("name", ["../escape", "/absolute", "xl/vbaProject.bin", "xl/externalLinks/link.xml"])
def test_unsafe_archive_entries(name):
    output = io.BytesIO()
    with zipfile.ZipFile(output, "w") as archive:
        archive.writestr(name, "bad")
    with pytest.raises(ValueError):
        check_archive(output.getvalue())


def test_missing_sales_are_not_zero_and_incomplete_order_excluded(store, settings):
    data = ingest(
        store, settings, "x.csv", b"date,amount,order\n2025-01-01,10,A\n2025-01-01,,A\n2025-01-02,20,B\n"
    )
    save_mapping(
        store,
        data["id"],
        Mapping(
            sheet="CSV",
            columns={"transaction_date": "date", "sales_amount": "amount", "order_id": "order"},
            grain="line_item",
            currency="DEMO",
            sales_definition="Net sales after monetary discounts.",
            confirmed=True,
        ),
    )
    clean(store, data["id"], Cleaning(confirmed=True))
    result = analyze(store, data["id"])["result"]
    assert result["sales_rows_missing"] == 1 and result["average_order_value"] == 20
    assert result["incomplete_orders"] == 1


def test_gross_discount_and_per_unit_cost(store, settings):
    data = ingest(store, settings, "x.csv", b"date,amount,discount,cost,qty\n2025-01-01,100,10,20,2\n")
    save_mapping(
        store,
        data["id"],
        Mapping(
            sheet="CSV",
            columns={
                "transaction_date": "date",
                "sales_amount": "amount",
                "discount": "discount",
                "cost": "cost",
                "sales_quantity": "qty",
            },
            grain="line_item",
            currency="DEMO",
            sales_definition="Gross before monetary discount, no tax.",
            sales_basis="gross_before_discount",
            cost_basis="per_unit",
            confirmed=True,
        ),
    )
    clean(store, data["id"], Cleaning(confirmed=True))
    assert analyze(store, data["id"])["result"]["net_sales"] == 90
    assert analyze(store, data["id"])["result"]["profit"] == 50


def test_cache_changes_with_dataset_version(store, prepared):
    first = store.cache_path(prepared, "summary", {})
    clean(store, prepared, Cleaning(confirmed=True))
    assert first != store.cache_path(prepared, "summary", {})


def test_dashboard_explains_missing_analyses(store, prepared):
    result = dashboard(store, prepared)
    assert len(result["figures"]) == 3
    assert "basket" in result["unavailable"]


def test_export_neutralizes_spreadsheet_formula(store, prepared):
    frame, data = load_clean(store, prepared)
    frame.loc[0, "customer_id"] = "=1+1"
    frame.to_parquet(store.directory(prepared) / data["meta"]["cleaned"], index=False)
    body, _, _ = export(store, prepared, "csv")
    assert "'=1+1" in body.decode("utf-8-sig")
    xlsx, _, _ = export(store, prepared, "xlsx")
    book = openpyxl.load_workbook(io.BytesIO(xlsx))
    assert all(cell.data_type != "f" for row in book["Cleaned"] for cell in row)


def test_scenario_is_explicit_arithmetic(store, prepared):
    with pytest.raises(ValueError, match="Confirm"):
        simulate(store, prepared, Simulation(price_change=0.1))
    result = simulate(store, prepared, Simulation(price_change=0.1, assumptions_confirmed=True))
    assert result["hypothetical_net_sales"] == pytest.approx(275)
    assert result["causal_prediction"] is False


### Code cell · `tests/test_agents_ml.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'tests/test_agents_ml.py'}"
import json
from pathlib import Path

import pytest

from sales_intelligence.agents import run_agent
from sales_intelligence.config import Settings
from sales_intelligence.ingestion import ingest
from sales_intelligence.ml import cluster, forecast, supervised
from sales_intelligence.nlp import text_analysis
from sales_intelligence.preparation import clean, save_mapping
from sales_intelligence.rag import add_document, retrieve
from sales_intelligence.schemas import Chat, Cleaning, Forecast, Mapping, Supervised
from scripts.make_sample import create_sample


def test_agent_queries_real_data_and_cites_version(store, settings, prepared):
    result = run_agent(store, settings, prepared, Chat(question="Which products sell best?"))
    assert result["selected_tool"] == "products"
    evidence = result["verified_findings"][0]
    assert evidence["values"]["ranking"][0]["net_sales"] == 180
    assert evidence["version"] == store.get(prepared)["version"]
    assert all(
        evidence["id"] in item["evidence_ids"] for item in result["interpretations_and_review_actions"]
    )


def test_agent_does_not_fabricate_churn_or_ignore_relative_dates(store, settings, prepared):
    for question in ["Which customers will churn?", "فروش ماه گذشته چقدر بود؟"]:
        result = run_agent(store, settings, prepared, Chat(question=question))
        assert result["status"] == "needs_clarification_or_data"
        assert result["verified_findings"] == []


def test_agent_rejects_model_invented_evidence_ids(store, settings, prepared, monkeypatch):
    from sales_intelligence.llm import LocalModel
    from sales_intelligence.schemas import Advice, AdviceItem, Route

    monkeypatch.setattr(LocalModel, "route", lambda self, question: (Route(tool="products"), {}))
    monkeypatch.setattr(
        LocalModel,
        "advice",
        lambda self, evidence: (
            Advice(items=[AdviceItem(action="review_product_mix", evidence_ids=["invented-source"])]),
            {},
        ),
    )
    monkeypatch.setattr("sales_intelligence.agents.retrieve", lambda *args: [])
    result = run_agent(store, settings, prepared, Chat(question="Which products sell best?", use_llm=True))
    assert result["verified_findings"]
    assert result["interpretations_and_review_actions"] == []


@pytest.mark.parametrize(
    "url",
    [
        "https://api.example.com",
        "http://127.0.0.1@evil.example",
        "http://localhost:11434/redirect",
        "http://10.0.0.1",
    ],
)
def test_cloud_model_endpoints_rejected(url):
    with pytest.raises(ValueError):
        Settings(api_key="a" * 32, ollama_url=url, _env_file=None)


def test_no_text_reports_skip(store, prepared):
    assert text_analysis(store, prepared)["status"] == "skipped"


@pytest.fixture
def sample(store, settings, tmp_path):
    frame = create_sample(tmp_path / "fictional")
    data = ingest(store, settings, "sales.csv", frame.to_csv(index=False).encode())
    columns = {
        name: name
        for name in [
            "order_id",
            "customer_id",
            "product_id",
            "transaction_date",
            "sales_quantity",
            "sales_amount",
            "cost",
        ]
    }
    columns["text"] = "feedback"
    save_mapping(
        store,
        data["id"],
        Mapping(
            sheet="CSV",
            columns=columns,
            grain="line_item",
            currency="DEMO",
            sales_definition="Fictional net sales excluding tax and shipping.",
            cost_basis="line_total",
            confirmed=True,
        ),
    )
    clean(store, data["id"], Cleaning(confirmed=True))
    return data["id"]


def test_forecast_uses_past_only_and_compares_baseline(store, sample):
    result = forecast(store, sample, Forecast(horizon=7))
    assert len(result["forecast"]) == 7
    assert all(fold["train_end"] < fold["test_start"] for fold in result["folds"])
    assert result["model_mae"] >= 0 and result["baseline_mae"] >= 0
    if result["baseline_mae"] <= result["model_mae"]:
        assert result["model"] == "seasonal_naive_7"


def test_insufficient_data_refuses_ml(store, prepared):
    with pytest.raises(ValueError):
        forecast(store, prepared, Forecast())
    with pytest.raises(ValueError):
        cluster(store, prepared)


def test_supervised_requires_label_definition_and_leakage_review(store, sample):
    request = Supervised(
        task="classification",
        target="raw__risk_label",
        features=["sales_quantity"],
        target_definition="Synthetic demonstration label, never a valid churn label.",
        prediction_horizon="Demonstration only",
    )
    with pytest.raises(ValueError, match="review"):
        supervised(store, sample, request)
    request.leakage_reviewed = True
    request.features = ["raw__risk_label"]
    with pytest.raises(ValueError, match="target"):
        supervised(store, sample, request)


def test_supervised_and_clustering_run_on_fictional_data(store, sample):
    request = Supervised(
        task="classification",
        target="raw__risk_label",
        features=["sales_quantity"],
        target_definition="Synthetic fixture classification, not a real churn target.",
        prediction_horizon="Demonstration only",
        leakage_reviewed=True,
    )
    result = supervised(store, sample, request)
    assert "baseline_f1_macro" in result["metrics"]
    assert cluster(store, sample)["clusters"] >= 2
    assert text_analysis(store, sample)["status"] == "completed"


class FakeEmbeddings:
    class settings:
        embedding_model = "test-local-embedding"

    calls = 0

    def verify_downloaded(self, name):
        return "fixed-fixture-digest"

    def embed(self, texts):
        self.calls += 1
        return [
            [float("bundle" in text.lower()) + 0.01, float("consent" in text.lower()) + 0.01, 0.1]
            for text in texts
        ]


def test_rag_source_attribution_and_embedding_dedup(store, prepared):
    model = FakeEmbeddings()
    body = b"Bundles require manager review. Customer consent is required."
    first = add_document(store, prepared, "../../policy.txt", body, model)
    calls = model.calls
    second = add_document(store, prepared, "policy.txt", body, model)
    assert second["cached"] is True and first["document_id"] == second["document_id"]
    assert model.calls == calls
    result = retrieve(store, prepared, "bundle", model)
    assert result[0]["source"] == "policy.txt" and result[0]["source_id"].endswith(":0")


def test_workflow_graphs_have_no_secrets_and_have_matching_edges():
    for path in (Path(__file__).parents[1] / "workflows").glob("*.json"):
        flow = json.loads(path.read_text())
        names = {node["name"] for node in flow["nodes"]}
        assert flow["active"] is False
        assert all(node["type"] != "n8n-nodes-base.code" for node in flow["nodes"])
        for source, connections in flow["connections"].items():
            assert source in names
            assert all(edge["node"] in names for branch in connections["main"] for edge in branch)


### Code cell · `tests/test_api.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'tests/test_api.py'}"
import time

from fastapi.testclient import TestClient

from sales_intelligence.api import create_app


def test_auth_and_path_validation(settings):
    app = create_app(settings)
    client = TestClient(app)
    assert client.get("/api/health").status_code == 401
    assert client.get("/api/jobs/not-a-uuid", headers={"X-API-Key": settings.api_key}).status_code == 422
    assert client.get("/api/health", headers={"X-API-Key": settings.api_key}).status_code == 200


def test_end_to_end_upload_approval_background_analysis_export(settings):
    with TestClient(create_app(settings)) as client:
        client.headers["X-API-Key"] = settings.api_key
        response = client.post(
            "/api/upload",
            files={"file": ("s.csv", b"date,amount,order\n2025-01-01,100,A\n2025-01-02,50,B\n", "text/csv")},
        )
        assert response.status_code == 201, response.text
        dataset_id = response.json()["id"]
        assert client.post("/api/analyze/" + dataset_id, json={}).status_code == 400
        mapping = {
            "sheet": "CSV",
            "columns": {"transaction_date": "date", "sales_amount": "amount", "order_id": "order"},
            "grain": "order",
            "currency": "DEMO",
            "sales_definition": "Net sales excluding tax and shipping.",
            "confirmed": True,
        }
        assert client.post("/api/mapping/" + dataset_id, json=mapping).status_code == 200
        assert client.post("/api/clean/" + dataset_id, json={"confirmed": True}).status_code == 200
        receipt = client.post("/api/analyze/" + dataset_id, json={})
        assert receipt.status_code == 202, receipt.text
        task_id = receipt.json()["task_id"]
        assert client.post("/api/analyze/" + dataset_id, json={}).json()["task_id"] == task_id
        for _ in range(200):
            task = client.get("/api/tasks/" + task_id).json()
            if task["status"] in {"completed", "failed"}:
                break
            time.sleep(0.05)
        assert task["status"] == "completed", task
        assert task["result"]["output"]["analyses"]["summary"]["result"]["net_sales"] == 150
        assert client.get("/api/dashboard/" + dataset_id).status_code == 200
        assert client.get("/api/export/" + dataset_id + "?format=xlsx").status_code == 200
        assert client.get("/api/audit/" + dataset_id).json()


### Code cell · `tests/test_text_and_safety.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'tests/test_text_and_safety.py'}"
import io
from datetime import datetime

import openpyxl
import pandas as pd
import pytest

from sales_intelligence.ingestion import ingest
from sales_intelligence.nlp import evaluate_text_classifier
from sales_intelligence.preparation import clean, save_mapping
from sales_intelligence.schemas import Cleaning, Mapping, TextClassifier


def test_native_excel_dates_work_with_confirmed_date_format(store, settings):
    book = openpyxl.Workbook()
    sheet = book.active
    sheet.title = "Sales"
    sheet.append(["date", "sales"])
    sheet.append([datetime(2025, 1, 1), 100])
    buffer = io.BytesIO()
    book.save(buffer)
    data = ingest(store, settings, "sales.xlsx", buffer.getvalue())
    save_mapping(
        store,
        data["id"],
        Mapping(
            sheet="Sales",
            columns={"transaction_date": "date", "sales_amount": "sales"},
            grain="order",
            currency="DEMO",
            sales_definition="Net sales excluding tax and shipping.",
            confirmed=True,
        ),
    )
    assert clean(store, data["id"], Cleaning(confirmed=True))["issues"] == []


def test_sentiment_evaluation_requires_existing_reviewed_labels(store, settings):
    dates = pd.date_range("2025-01-01", periods=120)
    rows = [
        {
            "date": date.date().isoformat(),
            "amount": 10,
            "feedback": ("excellent quality happy " if index % 2 else "bad delivery unhappy ")
            + f"reference{index}",
            "sentiment": "positive" if index % 2 else "negative",
        }
        for index, date in enumerate(dates)
    ]
    data = ingest(store, settings, "labeled.csv", pd.DataFrame(rows).to_csv(index=False).encode())
    save_mapping(
        store,
        data["id"],
        Mapping(
            sheet="CSV",
            columns={"transaction_date": "date", "sales_amount": "amount", "text": "feedback"},
            grain="order",
            currency="DEMO",
            sales_definition="Fictional net amounts for a text classification test.",
            confirmed=True,
        ),
    )
    clean(store, data["id"], Cleaning(confirmed=True))
    request = TextClassifier(
        purpose="sentiment",
        label_column="raw__sentiment",
        label_definition="Synthetic fixture sentiment, not a domain evaluation.",
        permitted_labels=["positive", "negative"],
    )
    with pytest.raises(ValueError, match="review"):
        evaluate_text_classifier(store, data["id"], request)
    request.labeling_reviewed = True
    result = evaluate_text_classifier(store, data["id"], request)
    assert result["test_rows"] >= 20
    assert result["f1_macro"] > result["baseline_f1_macro"]
    assert result["evaluation_deduplicated_texts"] is True


### Code cell · `scripts/benchmark.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/benchmark.py'}"
"""Measure the same fictional workload locally; do not infer production performance."""

import json
import threading
import time
from pathlib import Path

import psutil

from sales_intelligence.analytics import dashboard
from sales_intelligence.ml import forecast
from sales_intelligence.schemas import Forecast
from scripts.notebook_demo import make_demo

ROOT = Path(__file__).resolve().parents[1]


def main():
    process = psutil.Process()
    peak = [process.memory_info().rss]
    done = threading.Event()

    def sample():
        while not done.wait(0.02):
            peak[0] = max(peak[0], process.memory_info().rss)

    thread = threading.Thread(target=sample, daemon=True)
    thread.start()
    durations = {}
    start = time.perf_counter()
    store, _settings, dataset_id, frame = make_demo(ROOT)
    durations["ingestion_mapping_cleaning_seconds"] = time.perf_counter() - start
    for name, operation in [
        ("analysis_dashboard", lambda: dashboard(store, dataset_id)),
        ("forecast_training_validation", lambda: forecast(store, dataset_id, Forecast(horizon=7))),
    ]:
        start = time.perf_counter()
        operation()
        durations[name + "_seconds"] = time.perf_counter() - start
    done.set()
    thread.join()
    result = {
        "fictional_rows": len(frame),
        "durations": durations,
        "sampled_peak_rss_bytes": peak[0],
        "logical_cpus": psutil.cpu_count(),
        "cpu_times": process.cpu_times()._asdict(),
        "not_measured": ["LLM latency", "live n8n execution", "production workload"],
        "note": "Observed RSS sampled every 20 ms; may miss shorter spikes. Fixture generation is included in preparation timing.",
    }
    (ROOT / ".cache").mkdir(exist_ok=True)
    (ROOT / ".cache/benchmark.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()


### Code cell · `scripts/verify_notebooks.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/verify_notebooks.py'}"
"""Validate every notebook and execute the master with a private project-local kernel."""

import json
import os
import sys
from pathlib import Path

import nbformat
from nbclient import NotebookClient

ROOT = Path(__file__).resolve().parents[1]
cache = ROOT / ".cache"
kernel = cache / "jupyter/kernels/sales-check"
kernel.mkdir(parents=True, exist_ok=True)
(kernel / "kernel.json").write_text(
    json.dumps(
        {
            "argv": [sys.executable, "-m", "ipykernel_launcher", "-f", "{connection_file}"],
            "display_name": "Sales verification",
            "language": "python",
        }
    )
)
for variable, suffix in [
    ("JUPYTER_PATH", "jupyter"),
    ("JUPYTER_RUNTIME_DIR", "runtime"),
    ("IPYTHONDIR", "ipython"),
    ("MPLCONFIGDIR", "matplotlib"),
]:
    directory = cache / suffix
    directory.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(directory)
notebooks = sorted((ROOT / "notebooks").glob("*.ipynb"))
for path in notebooks:
    notebook = nbformat.read(path, as_version=4)
    nbformat.validate(notebook)
    assert any(cell.cell_type == "code" for cell in notebook.cells)
print(f"Validated {len(notebooks)} notebook schemas.", flush=True)
path = ROOT / "notebooks/00_Local_AI_Sales_Intelligence.ipynb"
notebook = nbformat.read(path, as_version=4)
client = NotebookClient(
    notebook, timeout=240, kernel_name="sales-check", resources={"metadata": {"path": str(ROOT)}}
)
client.execute()
executed = ROOT / ".cache/00_Local_AI_Sales_Intelligence.executed.ipynb"
nbformat.write(notebook, executed)
errors = [
    output
    for cell in notebook.cells
    if cell.cell_type == "code"
    for output in cell.get("outputs", [])
    if output.output_type == "error"
]
assert not errors, errors
print(
    f"Executed {sum(cell.cell_type == 'code' for cell in notebook.cells)} master notebook code cells without errors.",
    flush=True,
)
print(f"Executed copy: {executed}")


### Code cell · `scripts/package_project.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/package_project.py'}"
"""Create a source/notebook bundle without secrets, virtual environments or user data."""

import argparse
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
parser = argparse.ArgumentParser()
parser.add_argument("--output", type=Path, default=Path("/tmp/local-ai-sales-intelligence.zip"))
arguments = parser.parse_args()
files = []
for directory in [
    "sales_intelligence",
    "scripts",
    "tests",
    "notebooks",
    "workflows",
    "docs",
    "sample_data",
    "frontend",
]:
    for path in (ROOT / directory).rglob("*"):
        relative = path.relative_to(ROOT)
        if path.is_file() and not any(
            part in {"node_modules", "dist", "__pycache__", ".ipynb_checkpoints"} for part in relative.parts
        ):
            files.append(path)
for name in [
    "README.md",
    "pyproject.toml",
    "requirements.lock.txt",
    "requirements-dev.lock.txt",
    ".env.example",
    ".gitignore",
    ".dockerignore",
    "Dockerfile",
    "compose.yaml",
]:
    files.append(ROOT / name)
with zipfile.ZipFile(arguments.output, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(files):
        archive.write(path, Path("local-ai-sales-intelligence") / path.relative_to(ROOT))
with zipfile.ZipFile(arguments.output) as archive:
    assert archive.testzip() is None
    assert all(Path(name).name != ".env" for name in archive.namelist())
print(f"Packaged {len(files)} files: {arguments.output.resolve()}")


### Code cell · `scripts/build_notebooks.py`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'scripts/build_notebooks.py'}"
"""Build the teaching notebooks from the exact tested source files."""

from pathlib import Path

import nbformat as nbf

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "notebooks"

SETUP = """from pathlib import Path
import os, sys, json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the supplied project or its notebooks directory."
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Source-writing cells reconstruct a separate copy and preserve the supplied project.
BUILD_ROOT = ROOT / "notebook_project"
BUILD_ROOT.mkdir(exist_ok=True)
for folder in ["sales_intelligence", "scripts", "tests", "workflows", "frontend/src", "docs"]:
    (BUILD_ROOT / folder).mkdir(parents=True, exist_ok=True)
RUN_LOCAL_MODELS = False  # opt in only after installing local Ollama models
RUN_N8N = False          # opt in only after configuring/publishing the local workflows
print("Python:", sys.version.split()[0])
print("Source reconstruction:", BUILD_ROOT)
"""

DEMO = """if "demo_store" not in globals():
    from scripts.notebook_demo import make_demo
    demo_store, demo_settings, demo_id, sample_frame = make_demo(ROOT)
print("Fictional dataset:", demo_id)
"""

CHAPTERS = [
    (
        1,
        "Foundation",
        "Create the project configuration, SQLite metadata store and bounded file reader. Original files stay separate from parsed and cleaned versions. The supplied implementation targets one local company with an authenticated API.",
        [
            "pyproject.toml",
            ".gitignore",
            ".env.example",
            "sales_intelligence/__init__.py",
            "sales_intelligence/config.py",
            "sales_intelligence/schemas.py",
            "sales_intelligence/store.py",
            "sales_intelligence/ingestion.py",
            "scripts/bootstrap.py",
            "scripts/make_sample.py",
        ],
        """from tempfile import TemporaryDirectory
from sales_intelligence.config import Settings
from sales_intelligence.store import Store
from sales_intelligence.ingestion import ingest
with TemporaryDirectory() as directory:
    settings = Settings(api_key="fictional-notebook-key-" + "x"*32, data_dir=directory, _env_file=None).prepare()
    store = Store(settings.data_dir)
    dataset = ingest(store, settings, "tiny.csv", b"date,amount\\n2025-01-01,100\\n")
    assert dataset["meta"]["sheets"]["CSV"]["rows"] == 1
    print(dataset["meta"])
""",
        "Check that the original file and parsed table exist independently. CSV input is UTF-8. Formula-bearing workbooks are rejected; export calculated values before uploading. Review limits before importing large files.",
    ),
    (
        2,
        "Data intelligence",
        "Inspect every sheet, propose header matches, confirm the meaning of fields, validate a dimension join and record cleaning changes. This chapter uses deterministic mapping suggestions; local AI becomes optional in later chapters.",
        ["sales_intelligence/preparation.py"],
        DEMO
        + """from sales_intelligence.preparation import inspect, load_clean
profile = inspect(demo_store, demo_id)
cleaned, metadata = load_clean(demo_store, demo_id)
assert len(cleaned) == len(sample_frame)
assert metadata["meta"]["quality"]["removed_source_rows"] == []
display(profile)
display(cleaned.head())
""",
        "Human confirmation is required by the Mapping and Cleaning request schemas. Unknown identifiers stay unknown. One many-to-one or one-to-one dimension is supported; many-to-many joins are rejected. Review the change audit and missing-value coverage.",
    ),
    (
        3,
        "Core analytics and dashboard",
        "Calculate net sales, complete-order average value, product/customer rankings, RFM, monthly retention, basket associations, anomalies and correlations. Build Plotly figures from computed values.",
        [
            "sales_intelligence/analytics.py",
            "frontend/package.json",
            "frontend/package-lock.json",
            "frontend/tsconfig.json",
            "frontend/vite.config.ts",
            "frontend/index.html",
            "frontend/src/plotly.d.ts",
            "frontend/src/main.tsx",
            "frontend/src/style.css",
            "frontend/src/App.tsx",
        ],
        DEMO
        + """from sales_intelligence.analytics import analyze, dashboard
from sales_intelligence.schemas import Filters
totals = analyze(demo_store, demo_id)
assert totals["result"]["net_sales"] == float(sample_frame.sales_amount.sum())
display(totals)
charts = dashboard(demo_store, demo_id)
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML
display(HTML(pio.to_html(go.Figure(charts["figures"][0]), include_plotlyjs=True, full_html=False)))
display(analyze(demo_store, demo_id, "retention")["result"])
""",
        "Check units, filters, date coverage and order grain. Basket rules are associations. Retention uses first purchase observed in the supplied history. The React dashboard runs separately from Jupyter; launch instructions are in README.md.",
    ),
    (
        4,
        "Machine learning",
        "Evaluate a daily forecast against a seasonal baseline using expanding-window, recursive backtests. Run scaled RFM clustering and label-based supervised learning only when their requirements pass.",
        ["sales_intelligence/ml.py"],
        DEMO
        + """from sales_intelligence.ml import forecast, cluster, supervised
from sales_intelligence.schemas import Forecast, Supervised
forecast_result = forecast(demo_store, demo_id, Forecast(horizon=7))
assert all(fold["train_end"] < fold["test_start"] for fold in forecast_result["folds"])
display(forecast_result)
display(cluster(demo_store, demo_id))
# The sample risk label is explicitly synthetic. It demonstrates plumbing, not business validity.
model_result = supervised(demo_store, demo_id, Supervised(
    task="classification", target="raw__risk_label", features=["sales_quantity"],
    target_definition="Synthetic fixture label, never a genuine churn label.",
    prediction_horizon="Demonstration only", leakage_reviewed=True))
display(model_result)
""",
        "Inspect baseline performance before using a model. Forecast bands are exploratory residual bands, not guaranteed intervals. Churn, purchase propensity and CLV require defensible labels/definitions; this project does not invent them.",
    ),
    (
        5,
        "Local LLM engineering",
        "Connect one shared Ollama chat model through LangChain with validated JSON, bounded context, local-only endpoints, timeouts, limited retries and prompt hashes. The model selects tools and evidence-linked actions rather than authoring numbers.",
        ["sales_intelligence/llm.py"],
        DEMO
        + """from sales_intelligence.llm import LocalModel, PROMPTS
model = LocalModel(demo_settings, demo_store)
print("Registered role prompts:", sorted(PROMPTS))
with demo_store.db() as connection:
    print("Versioned prompts:", connection.execute("SELECT COUNT(*) FROM prompts").fetchone()[0])
if RUN_LOCAL_MODELS:
    from sales_intelligence.config import Settings
    local_settings = Settings().prepare()
    assert local_settings.use_llm, "Enable SALES_USE_LLM after downloading the models."
    local_model = LocalModel(local_settings)
    display(local_model.route("Which products have the highest sales?"))
else:
    print("Live model call not run. Enable RUN_LOCAL_MODELS after local Ollama setup.")
""",
        "Run Ollama with OLLAMA_NO_CLOUD=1. Download qwen3:8b and nomic-embed-text before enabling model calls. Setup downloads need internet; business-data processing stays local. Smaller/larger models can be configured after benchmarking.",
    ),
    (
        6,
        "Agent workflow",
        "Compose the orchestrator, data, analytics, intelligence and reporting nodes with LangGraph. Human-confirmed preparation is a prerequisite. SQLite checkpoints preserve state; suggestions carry dataset evidence IDs.",
        ["sales_intelligence/agents.py"],
        DEMO
        + """from sales_intelligence.agents import run_agent
from sales_intelligence.schemas import Chat
answer = run_agent(demo_store, demo_settings, demo_id, Chat(question="کدام محصولات بیشترین فروش را دارند؟"))
assert answer["status"] == "completed"
assert answer["verified_findings"][0]["values"]["ranking"]
display(answer)
unsupported = run_agent(demo_store, demo_settings, demo_id, Chat(question="Which customers will churn?"))
assert unsupported["status"] == "needs_clarification_or_data"
display(unsupported)
""",
        "Inspect the selected tool and its source/version. Relative dates require explicit UI date filters. Free-form SQL and generated Python execution are not provided. Checkpoints retain history but failed tasks restart explicitly; they do not resume model training mid-operation.",
    ),
    (
        7,
        "n8n automation",
        "Use n8n to submit a sales question and retrieve task status. A separate inactive schedule can submit a daily dashboard job. The backend owns data processing and agent state; n8n owns triggers and API connections.",
        [
            "scripts/build_workflows.py",
            "workflows/sales_advisor.json",
            "workflows/task_status.json",
            "workflows/daily_report.json",
        ],
        """workflows = {}
for path in (ROOT / "workflows").glob("*.json"):
    flow = json.loads(path.read_text())
    assert flow["active"] is False
    workflows[path.name] = {"name": flow["name"], "nodes": [node["name"] for node in flow["nodes"]]}
display(workflows)
if RUN_N8N:
    import getpass, httpx
    base = "http://localhost:5678"
    webhook_key = getpass.getpass("Local n8n webhook key: ")
    approved_dataset_id = input("Prepared API dataset UUID, not the notebook demonstration UUID: ").strip()
    response = httpx.post(base + "/webhook/sales-advisor", headers={"X-Webhook-Key": webhook_key},
        json={"dataset_id": approved_dataset_id, "question": "Which products sell best?", "use_llm": False},
        timeout=20, trust_env=False)
    response.raise_for_status()
    display(response.json())
else:
    print("Live n8n call not run. Import the JSON files and configure header credentials using docs/n8n_setup.md.")
""",
        "Use separate credentials for n8n webhook access and the backend API. Never export secrets into workflow JSON. Docker URLs use http://api:8000; a fully native setup uses http://127.0.0.1:8000. No workflow sends messages or publishes itself.",
    ),
    (
        8,
        "Local RAG and text analysis",
        "Extract bounded TXT/MD, text PDF or DOCX content, chunk it, embed through local Ollama and index vectors using FAISS. Document/model hashes prevent redundant embedding. Use deterministic TF-IDF and NMF for text exploration.",
        ["sales_intelligence/rag.py", "sales_intelligence/nlp.py"],
        DEMO
        + """from sales_intelligence.nlp import text_analysis
from sales_intelligence.rag import add_document, retrieve
display(text_analysis(demo_store, demo_id))
if RUN_LOCAL_MODELS:
    from sales_intelligence.config import Settings
    from sales_intelligence.llm import LocalModel
    local_model = LocalModel(Settings().prepare())
    policy = (ROOT / "sample_data/company_policy.txt").read_bytes()
    display(add_document(demo_store, demo_id, "fictional_policy.txt", policy, local_model))
    display(retrieve(demo_store, demo_id, "What approval is required for product bundles?", local_model))
else:
    print("Live embedding/retrieval not run. Tests exercise FAISS indexing with explicit fixture embeddings.")
""",
        "Sentiment and complaint classification are reported as not estimated without reviewed labels. evaluate_text_classifier trains and evaluates the supplied taxonomy with TF-IDF, logistic regression and a chronological holdout; it never invents labels. NMF topics are exploratory. Scanned PDFs need OCR outside this implementation. Retrieved documents are not treated as verified transaction facts.",
    ),
    (
        9,
        "Conversational API and background jobs",
        "Expose authenticated FastAPI endpoints for uploading, preparation approvals, queries, exports and background ML/RAG jobs. A bounded process pool handles longer operations; clients poll task IDs and can request cancellation or one retry.",
        ["sales_intelligence/jobs.py", "sales_intelligence/api.py", "scripts/serve.py"],
        DEMO
        + """from fastapi.testclient import TestClient
from sales_intelligence.api import create_app
# This in-process check starts no public server and uses only fictional credentials.
app = create_app(demo_settings)
client = TestClient(app)
assert client.get("/api/health").status_code == 401
client.headers["X-API-Key"] = demo_settings.api_key
assert client.get("/api/health").status_code == 200
display(client.get("/api/schema/" + demo_id).json())
""",
        "Run one API server worker; its internal process pool is configurable. Cancellation is cooperative at task boundaries. A restart marks incomplete jobs failed for explicit retry. Dataset changes invalidate queued and in-flight results.",
    ),
    (
        10,
        "Simulation, reporting, deployment and verification",
        "Calculate explicit arithmetic scenarios, export safe CSV/XLSX/JSON/HTML, document assumptions, package local deployment and run the full acceptance tests. Cleanup previews expire-after-inactivity candidates before explicit deletion.",
        [
            "sales_intelligence/reporting.py",
            "scripts/cleanup.py",
            "scripts/notebook_demo.py",
            "Dockerfile",
            ".dockerignore",
            "compose.yaml",
            "frontend/Dockerfile",
            "frontend/nginx.conf",
            "tests/conftest.py",
            "tests/test_data.py",
            "tests/test_agents_ml.py",
            "tests/test_api.py",
            "tests/test_text_and_safety.py",
            "scripts/benchmark.py",
            "scripts/verify_notebooks.py",
            "scripts/package_project.py",
            "scripts/build_notebooks.py",
            "requirements.lock.txt",
            "requirements-dev.lock.txt",
            "README.md",
            "docs/architecture.md",
            "docs/n8n_setup.md",
            "docs/technical_report.md",
            "docs/sources.md",
            "docs/validation.md",
        ],
        DEMO
        + """from sales_intelligence.reporting import simulate, export
from sales_intelligence.schemas import Simulation
scenario = simulate(demo_store, demo_id, Simulation(price_change=0.05, assumptions_confirmed=True))
assert scenario["causal_prediction"] is False
display(scenario)
for output_format in ["csv", "xlsx", "json", "html"]:
    body, mime, name = export(demo_store, demo_id, output_format)
    assert len(body) > 0
    print(name, mime, len(body), "bytes")
# Run the tests in a terminal, or opt in here. This takes longer than the demonstration cells.
RUN_TESTS = False
if RUN_TESTS:
    import subprocess
    subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, check=True)
""",
        "Review docs/technical_report.md for delivered capabilities and limits. No local model, n8n server, Docker deployment or user-facing service is started by Run All. Integration cells stay opt-in. Do not use synthetic labels as evidence of predictive business value.",
    ),
]


def md(text):
    return nbf.v4.new_markdown_cell(text)


def code(text):
    return nbf.v4.new_code_cell(text.rstrip() + "\n")


def source_cells(paths):
    cells = []
    for path in paths:
        full = ROOT / path
        if not full.is_file():
            raise FileNotFoundError(full)
        cells.append(
            md(
                f"### Code cell · `{path}`\n\nThis is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`."
            )
        )
        cells.append(code(f'%%writefile "{{BUILD_ROOT / {path!r}}}"\n' + full.read_text()))
    return cells


def write_notebook(path, cells):
    notebook = nbf.v4.new_notebook(
        cells=cells,
        metadata={
            "kernelspec": {"display_name": "Python 3 (local-sales)", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.13"},
        },
    )
    nbf.validate(notebook)
    nbf.write(notebook, path)


def build():
    OUT.mkdir(exist_ok=True)
    introduction = """# Local AI Sales Intelligence\n\nA complete source walkthrough in ten chapters. Each chapter contains explanations, full file-writing code cells, runnable fictional-data examples and a verification checkpoint.\n\nStart Jupyter from this project using its virtual environment. Read `README.md` first. The delivered source files are already present. Source cells reconstruct a separate copy in `notebook_project/`; demonstrations import the supplied, tested package.\n\nAll demonstrations use clearly fictional transactions. Model and n8n calls are opt-in. `Run All` does not install system software, download models, publish workflows, or start services. To review one chapter at a time, use the individual notebooks and pause at each checkpoint.\n\nThe original prompt's ten topics are retained. Basic deterministic preparation comes before optional local models, and the complete web API is assembled after its modules exist.\n"""
    master = [md(introduction), md("## Cell 1 · Notebook setup"), code(SETUP)]
    index = [
        "# Notebook index",
        "",
        "Open `00_Local_AI_Sales_Intelligence.ipynb` for the entire project, or use one chapter at a time.",
        "",
    ]
    for number, title, description, paths, example, checkpoint in CHAPTERS:
        header = md(f"## Chapter {number} · {title}\n\n{description}")
        cells = [
            header,
            *source_cells(paths),
            md("### Run the chapter example"),
            code(example),
            md(
                "### Review checkpoint\n\n"
                + checkpoint
                + "\n\nReview this result before moving to the next chapter."
            ),
        ]
        slug = title.lower().replace(" ", "_").replace(",", "")
        filename = f"{number:02d}_{slug}.ipynb"
        write_notebook(OUT / filename, [md(introduction), code(SETUP), *cells])
        master.extend(cells)
        index.append(f"{number}. [{title}]({filename})")
    write_notebook(OUT / "00_Local_AI_Sales_Intelligence.ipynb", master)
    (OUT / "README.md").write_text("\n".join(index) + "\n")
    print(
        f"Built 11 notebooks. Master has {len(master)} cells, including {sum(c.cell_type == 'code' for c in master)} code cells."
    )


if __name__ == "__main__":
    build()


### Code cell · `requirements.lock.txt`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'requirements.lock.txt'}"
# This file was autogenerated by uv via the following command:
#    uv --cache-dir /tmp/sales-intelligence-uv-cache pip compile pyproject.toml --python-version 3.13 --offline --output-file requirements.lock.txt
aiosqlite==0.22.1
    # via langgraph-checkpoint-sqlite
annotated-doc==0.0.5
    # via fastapi
annotated-types==0.8.0
    # via pydantic
anyio==4.16.0
    # via
    #   httpx
    #   httpx2
    #   langsmith
    #   starlette
certifi==2026.10.23
    # via
    #   httpcore
    #   httpx
    #   langchain-core
    #   requests
charset-normalizer==3.5.2
    # via requests
click==8.5.0
    # via uvicorn
cloudpickle==3.1.2
    # via joblib
distro==1.9.0
    # via langsmith
et-xmlfile==2.0.0
    # via openpyxl
faiss-cpu==1.15.1
    # via local-ai-sales-intelligence (pyproject.toml)
fastapi==0.162.0
    # via local-ai-sales-intelligence (pyproject.toml)
formulaic==1.2.2
    # via statsmodels
h11==0.16.0
    # via
    #   httpcore
    #   httpcore2
    #   uvicorn
httpcore==1.0.9
    # via httpx
httpcore2==2.16.0
    # via httpx2
httpx==0.28.1
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   langchain-core
    #   langgraph-sdk
    #   ollama
httpx2==2.16.0
    # via langsmith
idna==3.20
    # via
    #   anyio
    #   httpx
    #   httpx2
    #   requests
interface-meta==2.0.1
    # via formulaic
jalali-core==1.0.0
    # via jdatetime
jdatetime==5.3.0
    # via local-ai-sales-intelligence (pyproject.toml)
joblib==1.6.0
    # via scikit-learn
jsonpatch==1.41
    # via langchain-core
jsonpointer==3.8
    # via jsonpatch
langchain-core==1.7.4
    # via
    #   langchain-ollama
    #   langgraph
    #   langgraph-checkpoint
    #   langgraph-prebuilt
    #   langgraph-sdk
langchain-ollama==1.1.0
    # via local-ai-sales-intelligence (pyproject.toml)
langchain-protocol==0.0.20
    # via
    #   langchain-core
    #   langgraph-sdk
langgraph==1.2.19
    # via local-ai-sales-intelligence (pyproject.toml)
langgraph-checkpoint==4.3.1
    # via
    #   langgraph
    #   langgraph-checkpoint-sqlite
    #   langgraph-prebuilt
langgraph-checkpoint-sqlite==3.1.3
    # via local-ai-sales-intelligence (pyproject.toml)
langgraph-prebuilt==1.1.1
    # via langgraph
langgraph-sdk==0.4.11
    # via langgraph
langsmith==0.14.17
    # via langchain-core
lxml==6.1.3
    # via python-docx
narwhals==2.28.0
    # via
    #   formulaic
    #   plotly
    #   scikit-learn
numpy==2.5.4
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   faiss-cpu
    #   formulaic
    #   pandas
    #   patsy
    #   scikit-learn
    #   scipy
    #   statsmodels
olefile==0.47
    # via local-ai-sales-intelligence (pyproject.toml)
ollama==0.6.3
    # via langchain-ollama
openpyxl==3.1.5
    # via local-ai-sales-intelligence (pyproject.toml)
opentelemetry-api==1.45.2
    # via fastapi
orjson==3.13.0
    # via
    #   langgraph-sdk
    #   langsmith
ormsgpack==1.13.0
    # via langgraph-checkpoint
packaging==26.3
    # via
    #   faiss-cpu
    #   langchain-core
    #   langsmith
    #   patsy
    #   plotly
    #   statsmodels
pandas==3.1.0
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   formulaic
    #   statsmodels
patsy==1.0.3
    # via statsmodels
plotly==6.9.0
    # via local-ai-sales-intelligence (pyproject.toml)
psutil==7.2.2
    # via local-ai-sales-intelligence (pyproject.toml)
pyarrow==24.0.0
    # via local-ai-sales-intelligence (pyproject.toml)
pydantic==2.14.2
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   fastapi
    #   langchain-core
    #   langgraph
    #   langsmith
    #   ollama
    #   pydantic-settings
pydantic-core==2.50.2
    # via pydantic
pydantic-settings==2.16.2
    # via local-ai-sales-intelligence (pyproject.toml)
pypdf==6.21.4
    # via local-ai-sales-intelligence (pyproject.toml)
python-dateutil==4.0.0
    # via pandas
python-docx==1.2.0
    # via local-ai-sales-intelligence (pyproject.toml)
python-dotenv==1.2.4
    # via pydantic-settings
python-multipart==0.0.33
    # via local-ai-sales-intelligence (pyproject.toml)
pyyaml==6.0.3
    # via langchain-core
requests==2.34.2
    # via
    #   langsmith
    #   requests-toolbelt
requests-toolbelt==1.0.0
    # via langsmith
scikit-learn==1.9.1
    # via local-ai-sales-intelligence (pyproject.toml)
scipy==1.18.1
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   formulaic
    #   scikit-learn
    #   statsmodels
sniffio==1.3.1
    # via langsmith
sqlite-vec==0.1.9
    # via langgraph-checkpoint-sqlite
starlette==1.10.0
    # via fastapi
statsmodels==0.15.0
    # via local-ai-sales-intelligence (pyproject.toml)
tenacity==9.2.1
    # via langchain-core
threadpoolctl==3.7.0
    # via scikit-learn
truststore==0.10.5
    # via
    #   httpcore2
    #   httpx2
typing-extensions==4.16.0
    # via
    #   anyio
    #   fastapi
    #   formulaic
    #   langchain-core
    #   langchain-protocol
    #   langsmith
    #   opentelemetry-api
    #   pydantic
    #   pydantic-core
    #   pydantic-settings
    #   python-docx
    #   typing-inspection
typing-inspection==0.4.4
    # via
    #   fastapi
    #   pydantic
    #   pydantic-settings
tzdata==2026.6
    # via python-dateutil
urllib3==2.8.0
    # via requests
uuid-utils==0.19.0
    # via
    #   langchain-core
    #   langsmith
uvicorn==0.54.0
    # via local-ai-sales-intelligence (pyproject.toml)
websockets==17.4
    # via
    #   langgraph-sdk
    #   langsmith
wrapt==2.5.2
    # via formulaic
xlrd==2.0.2
    # via local-ai-sales-intelligence (pyproject.toml)
xxhash==4.0.2
    # via
    #   langgraph
    #   langsmith
zstandard==0.25.0
    # via langsmith


### Code cell · `requirements-dev.lock.txt`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'requirements-dev.lock.txt'}"
# This file was autogenerated by uv via the following command:
#    uv --cache-dir /tmp/sales-intelligence-uv-cache pip compile pyproject.toml --python-version 3.13 --extra dev --offline --output-file requirements-dev.lock.txt
aiosqlite==0.22.1
    # via langgraph-checkpoint-sqlite
annotated-doc==0.0.5
    # via fastapi
annotated-types==0.8.0
    # via pydantic
anyio==4.16.0
    # via
    #   httpx
    #   httpx2
    #   jupyter-server
    #   langsmith
    #   starlette
argon2-cffi==25.1.0
    # via jupyter-server
argon2-cffi-bindings==26.1.0
    # via argon2-cffi
arrow==1.4.0
    # via isoduration
asttokens==3.0.2
    # via stack-data
async-lru==2.4.0
    # via jupyterlab
attrs==26.1.0
    # via
    #   jsonschema
    #   referencing
babel==2.18.0
    # via jupyterlab-server
beautifulsoup4==4.15.0
    # via nbconvert
bleach==6.4.0
    # via nbconvert
certifi==2026.10.23
    # via
    #   httpcore
    #   httpx
    #   langchain-core
    #   requests
cffi==2.1.1
    # via argon2-cffi-bindings
charset-normalizer==3.5.2
    # via requests
click==8.5.0
    # via uvicorn
cloudpickle==3.1.2
    # via joblib
comm==0.2.3
    # via ipykernel
debugpy==1.8.22
    # via ipykernel
defusedxml==0.7.1
    # via nbconvert
distro==1.9.0
    # via langsmith
et-xmlfile==2.0.0
    # via openpyxl
executing==2.3.0
    # via stack-data
faiss-cpu==1.15.1
    # via local-ai-sales-intelligence (pyproject.toml)
fastapi==0.162.0
    # via local-ai-sales-intelligence (pyproject.toml)
fastjsonschema==2.22.3
    # via nbformat
formulaic==1.2.2
    # via statsmodels
fqdn==1.6.0
    # via jsonschema
h11==0.16.0
    # via
    #   httpcore
    #   httpcore2
    #   uvicorn
httpcore==1.0.9
    # via httpx
httpcore2==2.16.0
    # via httpx2
httpx==0.28.1
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   jupyterlab
    #   langchain-core
    #   langgraph-sdk
    #   ollama
httpx2==2.16.0
    # via langsmith
idna==3.20
    # via
    #   anyio
    #   httpx
    #   httpx2
    #   jsonschema
    #   requests
iniconfig==2.3.1
    # via pytest
interface-meta==2.0.1
    # via formulaic
ipykernel==7.4.0
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   jupyterlab
ipython==9.18.1
    # via ipykernel
ipython-pygments-lexers==1.1.1
    # via ipython
isoduration==20.11.0
    # via jsonschema
jalali-core==1.0.0
    # via jdatetime
jdatetime==5.3.0
    # via local-ai-sales-intelligence (pyproject.toml)
jedi==0.20.1
    # via ipython
jinja2==3.1.6
    # via
    #   jupyter-server
    #   jupyterlab
    #   jupyterlab-server
    #   nbconvert
joblib==1.6.0
    # via scikit-learn
json5==0.17.3
    # via jupyterlab-server
jsonpatch==1.41
    # via langchain-core
jsonpointer==3.8
    # via
    #   jsonpatch
    #   jsonschema
jsonschema==4.26.0
    # via
    #   jupyter-events
    #   jupyterlab-server
    #   nbformat
jsonschema-specifications==2025.9.1
    # via jsonschema
jupyter-builder==1.2.4
    # via jupyterlab
jupyter-client==8.10.0
    # via
    #   ipykernel
    #   jupyter-server
    #   nbclient
jupyter-core==5.9.1
    # via
    #   ipykernel
    #   jupyter-builder
    #   jupyter-client
    #   jupyter-server
    #   jupyterlab
    #   nbclient
    #   nbconvert
    #   nbformat
jupyter-events==0.12.1
    # via jupyter-server
jupyter-lsp==2.3.1
    # via jupyterlab
jupyter-server==2.21.1
    # via
    #   jupyter-lsp
    #   jupyterlab
    #   jupyterlab-server
    #   notebook-shim
jupyter-server-terminals==0.5.4
    # via jupyter-server
jupyterlab==4.6.5
    # via local-ai-sales-intelligence (pyproject.toml)
jupyterlab-pygments==0.3.0
    # via nbconvert
jupyterlab-server==2.28.1
    # via jupyterlab
langchain-core==1.7.4
    # via
    #   langchain-ollama
    #   langgraph
    #   langgraph-checkpoint
    #   langgraph-prebuilt
    #   langgraph-sdk
langchain-ollama==1.1.0
    # via local-ai-sales-intelligence (pyproject.toml)
langchain-protocol==0.0.20
    # via
    #   langchain-core
    #   langgraph-sdk
langgraph==1.2.19
    # via local-ai-sales-intelligence (pyproject.toml)
langgraph-checkpoint==4.3.1
    # via
    #   langgraph
    #   langgraph-checkpoint-sqlite
    #   langgraph-prebuilt
langgraph-checkpoint-sqlite==3.1.3
    # via local-ai-sales-intelligence (pyproject.toml)
langgraph-prebuilt==1.1.1
    # via langgraph
langgraph-sdk==0.4.11
    # via langgraph
langsmith==0.14.17
    # via langchain-core
lark==1.3.1
    # via rfc3987-syntax
lxml==6.1.3
    # via python-docx
markupsafe==3.0.4
    # via
    #   jinja2
    #   nbconvert
matplotlib-inline==0.2.2
    # via
    #   ipykernel
    #   ipython
mistune==3.4.0
    # via nbconvert
narwhals==2.28.0
    # via
    #   formulaic
    #   plotly
    #   scikit-learn
nbclient==0.11.0
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   nbconvert
nbconvert==7.17.2
    # via jupyter-server
nbformat==5.11.1
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   jupyter-server
    #   nbclient
    #   nbconvert
nest-asyncio2==1.7.4
    # via ipykernel
notebook-shim==0.2.4
    # via jupyterlab
numpy==2.5.4
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   faiss-cpu
    #   formulaic
    #   pandas
    #   patsy
    #   scikit-learn
    #   scipy
    #   statsmodels
olefile==0.47
    # via local-ai-sales-intelligence (pyproject.toml)
ollama==0.6.3
    # via langchain-ollama
openpyxl==3.1.5
    # via local-ai-sales-intelligence (pyproject.toml)
opentelemetry-api==1.45.2
    # via fastapi
orjson==3.13.0
    # via
    #   langgraph-sdk
    #   langsmith
ormsgpack==1.13.0
    # via langgraph-checkpoint
packaging==26.3
    # via
    #   faiss-cpu
    #   ipykernel
    #   jupyter-events
    #   jupyter-server
    #   jupyterlab
    #   jupyterlab-server
    #   langchain-core
    #   langsmith
    #   nbconvert
    #   patsy
    #   plotly
    #   pytest
    #   statsmodels
pandas==3.1.0
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   formulaic
    #   statsmodels
pandocfilters==1.5.1
    # via nbconvert
parso==0.8.7
    # via jedi
patsy==1.0.3
    # via statsmodels
pexpect==4.9.0
    # via ipython
platformdirs==4.14.3
    # via jupyter-core
plotly==6.9.0
    # via local-ai-sales-intelligence (pyproject.toml)
pluggy==1.7.0
    # via pytest
prometheus-client==0.26.0
    # via jupyter-server
prompt-toolkit==3.0.53
    # via ipython
psutil==7.2.2
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   ipython
ptyprocess==0.7.0
    # via
    #   pexpect
    #   terminado
pure-eval==0.2.4
    # via stack-data
pyarrow==24.0.0
    # via local-ai-sales-intelligence (pyproject.toml)
pycparser==3.12
    # via cffi
pydantic==2.14.2
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   fastapi
    #   langchain-core
    #   langgraph
    #   langsmith
    #   ollama
    #   pydantic-settings
pydantic-core==2.50.2
    # via pydantic
pydantic-settings==2.16.2
    # via local-ai-sales-intelligence (pyproject.toml)
pygments==2.21.0
    # via
    #   ipython
    #   ipython-pygments-lexers
    #   nbconvert
    #   pytest
pypdf==6.21.4
    # via local-ai-sales-intelligence (pyproject.toml)
pytest==9.2.0
    # via local-ai-sales-intelligence (pyproject.toml)
python-dateutil==4.0.0
    # via
    #   arrow
    #   jupyter-client
    #   pandas
python-docx==1.2.0
    # via local-ai-sales-intelligence (pyproject.toml)
python-dotenv==1.2.4
    # via pydantic-settings
python-json-logger==4.2.0
    # via jupyter-events
python-multipart==0.0.33
    # via local-ai-sales-intelligence (pyproject.toml)
pyyaml==6.0.3
    # via
    #   jupyter-events
    #   langchain-core
pyzmq==27.2.0
    # via
    #   ipykernel
    #   jupyter-client
    #   jupyter-server
referencing==0.37.0
    # via
    #   jsonschema
    #   jsonschema-specifications
    #   jupyter-events
requests==2.34.2
    # via
    #   jupyterlab-server
    #   langsmith
    #   requests-toolbelt
requests-toolbelt==1.0.0
    # via langsmith
rfc3339-validator==0.1.4
    # via
    #   jsonschema
    #   jupyter-events
rfc3986-validator==0.1.1
    # via
    #   jsonschema
    #   jupyter-events
rfc3987-syntax==1.1.0
    # via jsonschema
rpds-py==2026.9.1
    # via
    #   jsonschema
    #   referencing
ruff==0.17.3
    # via local-ai-sales-intelligence (pyproject.toml)
scikit-learn==1.9.1
    # via local-ai-sales-intelligence (pyproject.toml)
scipy==1.18.1
    # via
    #   local-ai-sales-intelligence (pyproject.toml)
    #   formulaic
    #   scikit-learn
    #   statsmodels
send2trash==2.1.0
    # via jupyter-server
six==1.17.0
    # via rfc3339-validator
sniffio==1.3.1
    # via langsmith
soupsieve==4.0
    # via beautifulsoup4
sqlite-vec==0.1.9
    # via langgraph-checkpoint-sqlite
stack-data==0.6.3
    # via ipython
starlette==1.10.0
    # via fastapi
statsmodels==0.15.0
    # via local-ai-sales-intelligence (pyproject.toml)
tenacity==9.2.1
    # via langchain-core
terminado==0.18.1
    # via
    #   jupyter-server
    #   jupyter-server-terminals
threadpoolctl==3.7.0
    # via scikit-learn
tinycss2==1.5.1
    # via bleach
tornado==6.6
    # via
    #   ipykernel
    #   jupyter-client
    #   jupyter-server
    #   jupyterlab
    #   terminado
traitlets==5.16.1
    # via
    #   ipykernel
    #   ipython
    #   jupyter-builder
    #   jupyter-client
    #   jupyter-core
    #   jupyter-events
    #   jupyter-server
    #   jupyterlab
    #   matplotlib-inline
    #   nbclient
    #   nbconvert
    #   nbformat
truststore==0.10.5
    # via
    #   httpcore2
    #   httpx2
typing-extensions==4.16.0
    # via
    #   anyio
    #   beautifulsoup4
    #   fastapi
    #   formulaic
    #   jupyter-client
    #   langchain-core
    #   langchain-protocol
    #   langsmith
    #   opentelemetry-api
    #   pydantic
    #   pydantic-core
    #   pydantic-settings
    #   python-docx
    #   typing-inspection
typing-inspection==0.4.4
    # via
    #   fastapi
    #   pydantic
    #   pydantic-settings
tzdata==2026.6
    # via
    #   arrow
    #   python-dateutil
uri-template==1.3.0
    # via jsonschema
urllib3==2.8.0
    # via requests
uuid-utils==0.19.0
    # via
    #   langchain-core
    #   langsmith
uvicorn==0.54.0
    # via local-ai-sales-intelligence (pyproject.toml)
wcwidth==0.9.4
    # via prompt-toolkit
webcolors==25.10.0
    # via jsonschema
webencodings==0.6.1
    # via
    #   bleach
    #   tinycss2
websocket-client==1.9.2
    # via jupyter-server
websockets==17.4
    # via
    #   langgraph-sdk
    #   langsmith
wrapt==2.5.2
    # via formulaic
xlrd==2.0.2
    # via local-ai-sales-intelligence (pyproject.toml)
xxhash==4.0.2
    # via
    #   langgraph
    #   langsmith
zstandard==0.25.0
    # via langsmith


### Code cell · `README.md`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'README.md'}"
# Local AI Sales Intelligence

A local sales-analysis application with a FastAPI backend, React/Plotly dashboard, LangGraph agent, local Ollama integration and importable n8n workflows. The complete source is also provided as Jupyter code cells in ten chapters.

Start with **[the complete notebook](notebooks/00_Local_AI_Sales_Intelligence.ipynb)**. Use [the chapter index](notebooks/README.md) to work through one topic at a time. Each chapter includes its full source files, a runnable example and a review checkpoint. Writing cells reconstruct a separate `notebook_project/` copy; they preserve the supplied source. Examples import the supplied package.

## Install and open Jupyter

Python 3.12–3.14 and Node.js 24 are supported. This checkout was validated with Python 3.13. The commands below work in Fish and Bash because they do not depend on shell activation.

```sh
uv venv --python 3.13 .venv
uv pip install --python .venv/bin/python -r requirements-dev.lock.txt
uv pip install --python .venv/bin/python --no-deps -e .
.venv/bin/python scripts/bootstrap.py
.venv/bin/python scripts/make_sample.py
.venv/bin/python -m jupyterlab notebooks/00_Local_AI_Sales_Intelligence.ipynb
```

If uv is unavailable, use `python3 -m venv .venv`, `.venv/bin/python -m pip install -r requirements-dev.lock.txt` and `.venv/bin/python -m pip install --no-deps -e .`. The locks record the validated Linux/Python 3.13 environment; on another platform or interpreter resolve from `pyproject.toml` with `.venv/bin/python -m pip install -e '.[dev]'`. On Windows, replace `.venv/bin/python` with `.venv/Scripts/python.exe`.

`bootstrap.py` creates a private `.env`, preserves an existing one and does not print secrets. The notebook examples use separate fictional credentials and data. Do not upload company data to a hosted notebook service. Initial package, image and model downloads require internet; runtime data processing uses local services.

## Run the native application

In the first terminal, from this directory:

```sh
.venv/bin/python scripts/serve.py
```

In a second terminal:

```sh
cd frontend
npm ci
npm run dev
```

Open [the local dashboard](http://127.0.0.1:5173). Copy the value of `SALES_API_KEY` from your local `.env` into its password field. The page keeps it only in memory. Upload `sample_data/sales.xlsx` or your own file, map the columns, describe what sales amounts mean, confirm the mapping, and approve the cleaning policy. Then calculate a dashboard or ask a question. Use the date filters for questions such as “last month.”

For the fictional sample, use `Sales`, one product per order row, currency `DEMO`, dates `%Y-%m-%d`, net sales after discount and line-total costs. Map `feedback` to text. Do not map the synthetic `risk_label` as a business outcome.

The local application works without an LLM. Its fallback recognises a limited set of Persian/English sales intents. Set `SALES_USE_LLM=true` only after setting up local Ollama. Supervised model evaluation and retrieval evaluation are available through Python/API and the notebooks. The dashboard provides the common analysis, forecast, clustering, text, document and scenario flows.

## Enable local models

Install Ollama separately. Run its server with cloud features disabled. For Fish:

```fish
env OLLAMA_NO_CLOUD=1 ollama serve
```

Download the configured chat and embedding models in another terminal:

```sh
ollama pull qwen3:8b
ollama pull nomic-embed-text
```

Update `.env` to `SALES_USE_LLM=true`, then restart the API. The application allows only loopback Ollama URLs or the `ollama` Compose service, verifies downloaded model tags, rejects cloud-tagged/remote-backed models and disables LangSmith tracing. Set the cloud-disable option on the Ollama server itself, including an existing server you operate separately. Hardware requirements depend on model quantization and context size; benchmark on your machine before choosing a larger model.

## Run with Docker Compose

```sh
.venv/bin/python scripts/bootstrap.py
docker compose up --build -d
docker compose exec ollama ollama pull qwen3:8b
docker compose exec ollama ollama pull nomic-embed-text
```

If enabling the LLM, edit `SALES_USE_LLM` in `.env` and recreate the API with `docker compose up -d api`. All published ports bind to `127.0.0.1`. Open the dashboard on port 5173 and n8n on port 5678. API data, Ollama models and n8n settings have separate persistent volumes. The Python server runs as an unprivileged container user. The native and Compose data stores are separate.

Compose pins n8n and Ollama image tags. Review image compatibility and update them deliberately. The Docker build uses `requirements.lock.txt` for the Python runtime. The frontend uses `package-lock.json` with `npm ci`.

## Import the n8n agent workflows

See [the n8n setup guide](docs/n8n_setup.md). Import:

- `workflows/sales_advisor.json` to submit an analysis question.
- `workflows/task_status.json` to retrieve results and evidence.
- `workflows/daily_report.json` for an optional daily local report submission.

All workflows are inactive. Importing the JSON does not configure credentials, publish a workflow or contact customers. Backend jobs are asynchronous. The submit webhook returns HTTP 202 and a task ID; the status webhook returns the result when ready.

## Validate

```sh
.venv/bin/python -m pytest -q
.venv/bin/ruff check sales_intelligence scripts tests
.venv/bin/python scripts/verify_notebooks.py
cd frontend
npm run build
```

`verify_notebooks.py` validates all notebooks and executes the master notebook with live model/n8n calls disabled. It uses the current Python environment as the kernel and keeps runtime/cache files inside this project. Run it after installing the dev dependencies. Regenerate notebooks after source changes with `.venv/bin/python scripts/build_notebooks.py`.

Read [the architecture](docs/architecture.md), [the technical report and limits](docs/technical_report.md), [the source references](docs/sources.md) and [the validation record](docs/validation.md).

## Retention

Datasets remain until explicit cleanup. `SALES_RETENTION_DAYS`, default 30, is an inactivity threshold used by the cleanup script. Preview with `.venv/bin/python scripts/cleanup.py`. Add `--apply` only when you intend to delete expired datasets, including originals and history. No destructive cleanup runs automatically. Protect local disks and backups because company files are stored locally without application-level encryption.


### Code cell · `docs/architecture.md`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'docs/architecture.md'}"
# Architecture

```mermaid
flowchart LR
  User[User] --> Web[React and Plotly]
  Web --> API[Authenticated FastAPI]
  N8N[n8n webhooks and schedule] --> API
  API --> Preparation[Inspect, map, join and clean]
  API --> Queue[Bounded process pool]
  Preparation --> Store[(SQLite and local files)]
  Queue --> Graph[LangGraph workflow]
  Graph --> Tools[Registered Python analyses]
  Graph --> Ollama[Local Ollama through LangChain]
  Graph --> RAG[Local FAISS retrieval]
  Tools --> Store
  Graph --> Evidence[Versioned findings and source IDs]
  Evidence --> API
```

## Responsibility boundaries

Python parses files, computes statistics, evaluates models and renders verified numbers. LangChain connects to the configured local model using Pydantic-validated output. LangGraph orders the agent roles and stores checkpoints. n8n starts jobs and queries the API. React provides preparation approvals, filters, charts, chat, exports and scenario controls. No component executes Python or SQL supplied by an LLM or uploaded file.

The five roles run sequentially through a single graph. A chat request chooses one registered analysis tool. Preparation is a separate, explicitly confirmed interaction. A submitted question cannot alter mappings, clean data, train an arbitrary algorithm or send messages. The model may select evidence-linked review actions from a fixed catalogue; Python supplies the numerical findings. This bounded design avoids passing unconstrained generated financial claims through to the user.

## Dataset lifecycle

1. Parse all accepted sheets once and preserve original bytes under a server-generated UUID.
2. Store raw sheets in Parquet and inspect column types, nulls and duplicates.
3. A person confirms canonical mappings, date format, currency, grain and sales/cost definitions.
4. Optionally confirm a dimension join. Unique non-null dimension keys prevent row multiplication.
5. Approve a cleaning policy. Write a separate cleaned Parquet file and per-cell change audit.
6. Increment the dataset version. Save analyses and tasks with their version and parameters.
7. Reject stale queued/in-flight results when the dataset changes.

Original client filenames never determine storage paths. Metadata updates use optimistic version checks. Content hashes identify source files. Analysis caches include dataset version, parameters and an implementation version. Updating mappings invalidates cleaning; a user must clean again.

## Jobs and persistence

One API process owns a bounded process pool. SQLite stores dataset/task metadata, audit events and versioned prompts. A separate SQLite database stores LangGraph checkpoints. Model loading is serial by default because the queue has one worker. If configuring two workers, use Ollama's one-model/one-request settings to limit inference contention. The in-process model mutex only serializes calls inside a worker.

Queue requests deduplicate by dataset, version, operation and parameters. Cancellation is checked before and after the operation, not inside every pandas or scikit-learn call. One explicit retry is allowed for a failed/cancelled analysis. An API restart marks unfinished jobs failed. Training does not resume halfway through an estimator. Job progress is coarse, and results include elapsed wall time, CPU time and resident RAM at completion.

## Business definitions

`sales_amount` is a confirmed amount for one line/order, never automatically inferred from `purchase_amount`. Net sales are either the confirmed net input or gross minus a confirmed monetary discount. Unit-price multiplication is not inferred. Order averages include only known order IDs with complete monetary values. Profit requires confirmed line costs or per-unit costs with quantity. Amounts use pandas numeric arithmetic and are analysis outputs, not a decimal accounting ledger.

Unknown dates/IDs remain unknown. Missing dates are excluded only where a time analysis requires them, and coverage is reported. Refunds can be negative net amounts. Basket analysis excludes nonpositive sales/quantities. Arithmetic scenarios reject negative net amounts and require complete input. Currency conversion is deliberately explicit; mixed/missing currency rows are rejected when a currency column is mapped.

## Local documents

Documents are scoped to a dataset. A hash of document bytes, embedding-model digest and chunk configuration identifies each immutable FAISS index. TXT/MD, text-based PDF and DOCX are supported. No FAISS pickle deserialization is used. Queries return source IDs, chunk indices and similarity scores. Similarity is not a probability of correctness. Relevance evaluation requires a human-labeled question/source set.

## Privacy and access

This version serves one trusted company on one host. A single API key protects every API route; it does not implement separate staff roles or tenant isolation. The frontend keeps the key in memory. n8n uses separate webhook and backend header credentials. Secrets are excluded from workflow exports and notebooks. Compose binds host ports to loopback and disables diagnostic calls. Do not expose this deployment to the internet without a separately designed authentication and transport layer.


### Code cell · `docs/n8n_setup.md`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'docs/n8n_setup.md'}"
# Set up the local n8n workflows

## Start services

Run `scripts/bootstrap.py`, then start the stack with `docker compose up --build -d`. Open [n8n](http://127.0.0.1:5678) and create its owner account. The application does not create an n8n login for you.

Import each JSON in `workflows/` through n8n's workflow import menu. The imports contain credential references named below; replace those references with credentials created in your own n8n instance. The IDs in the JSON are intentionally nonfunctional placeholders, not real credentials.

## Create two Header Auth credentials

1. Name one `Sales backend API`. Its header name is `X-API-Key`; its value is `SALES_API_KEY` from the private project `.env`. Assign it to each HTTP Request node.
2. Name the other `Sales webhook access`. Its header name is `X-Webhook-Key`; use a new random secret, different from the backend key. Assign it to both Webhook nodes.

Store these through n8n's credential UI. Do not paste secrets into expressions, JSON exports, chat messages or notebooks.

## Check local API URLs

Compose workflows use `http://api:8000`, which is the backend's container-network name. If n8n and the API both run natively, edit HTTP Request node URL origins to `http://127.0.0.1:8000`. If only n8n runs in Docker, configure a host-gateway route deliberately; container localhost points at n8n itself.

The request path uses `encodeURIComponent` for the dataset/task ID. The host is fixed and cannot be selected by webhook input. The backend validates UUIDs, request fields and preparation approvals.

## Prepare a dataset

Use the React dashboard to upload data and confirm its mapping and cleaning policy. Keep the dataset UUID shown by the dashboard. Notebook examples use a separate temporary dataset store, so their UUIDs do not refer to the API's store.

## Submit a question

Use the test URL while the Webhook node is listening. After configuring credentials, publish both the question and status workflows to enable the production URLs. In a Python notebook or script:

```python
import getpass
import time
import httpx

headers = {"X-Webhook-Key": getpass.getpass("n8n webhook key: ")}
dataset_id = input("Prepared dataset UUID: ").strip()
with httpx.Client(base_url="http://127.0.0.1:5678", trust_env=False, timeout=20) as client:
    response = client.post("/webhook/sales-advisor", headers=headers, json={
        "dataset_id": dataset_id,
        "question": "کدام محصولات بیشترین فروش را دارند؟",
        "filters": {},
        "use_llm": False,
    })
    response.raise_for_status()
    receipt = response.json()
    print("Task:", receipt["task_id"])
    for _ in range(300):
        response = client.get("/webhook/sales-status", headers=headers,
                              params={"task_id": receipt["task_id"]})
        response.raise_for_status()
        task = response.json()
        if task["status"] in {"completed", "failed", "cancelled"}:
            print(task)
            break
        time.sleep(1)
    else:
        print("Still running; use the task ID to check again later.")
```

The question webhook returns HTTP 202 and a task receipt. The result appears under `result.output` in task status. Verified findings, document excerpts and review suggestions have separate fields. A failed backend request stops the n8n execution; it does not silently turn an error into a successful recommendation. n8n execution payload saving is disabled in the shipped configuration.

## Optional daily report

Open `Local sales advisor - daily dashboard`. Replace `REPLACE_WITH_APPROVED_DATASET_UUID` in `Choose approved dataset`. Confirm the `Sales backend API` credential and the Asia/Tehran schedule, which is 08:00 daily. Publish only when you want scheduled execution. It queues a dashboard job and returns its receipt. It does not send emails, Telegram messages or other external notifications.

Because repeated requests with identical version/parameters reuse the existing result, the scheduled flow does not retrain or recalculate unchanged data unnecessarily. A newly uploaded dataset gets a new UUID; update the selected dataset deliberately. No directory watcher or unattended import is configured.

## Verify in your instance

Confirm the expected node types resolve, a sample question returns 202, the status workflow reaches `completed`, a bad webhook key is denied, and an invalid dataset does not produce a report. These checks require a running n8n instance. The repository's automated tests validate graph connections and the backend contract; they do not prove an n8n import or a live Ollama invocation succeeded.


### Code cell · `docs/technical_report.md`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'docs/technical_report.md'}"
# Technical report

## Delivered scope

The implementation is a bounded local application covering file ingestion, metadata/schema inspection, confirmed preparation, business analytics, evaluated ML, local structured model calls, a LangGraph agent, n8n job submission/status/scheduling, local document retrieval, text exploration, conversational tool selection, arithmetic scenarios and safe exports. All application source is included as Jupyter code cells. No mock sales result is used by the API; examples are explicitly fictional and isolated.

| Topic | Implementation and conditions |
|---|---|
| Excel/CSV | All ordinary worksheets; UTF-8 CSV; restricted non-macro XLS; XLSX archive/formula checks; 25 MB, 200,000 rows total and 200 columns by default |
| Mapping | Header-based suggestions, manual confirmation, explicit grain, calendar, currency and amount definitions |
| Relationships | One confirmed dimension join per dataset, validated many-to-one or one-to-one; ambiguous/unmatched keys require a decision |
| Cleaning | Preserve originals; separate cleaned versions; configurable numeric null/duplicate policy; invalid-value review; per-cell lineage |
| Analytics | Sales, time trends, customers/products/regions, AOV, discounts, Pareto, RFM, monthly cohorts, basket rules, MAD anomalies and numeric correlations |
| ML | Seasonal-baseline comparison, recursive daily forecast backtests, ridge regression, balanced logistic classification, RFM KMeans with silhouette comparison |
| Local model | Shared configurable Ollama model through LangChain, structured Pydantic output, bounded retries/context, prompt hashes, no cloud endpoints or external tracing |
| Agent | Five LangGraph roles; fixed tool registry, evidence references, human-confirmed preparation prerequisite, SQLite checkpoints |
| n8n | Three inactive importable workflows: async question submission, task status and optional scheduled dashboard |
| RAG | Local Ollama embeddings, FAISS, source attribution, byte/model/chunk hashes, text PDF/DOCX/TXT/MD, retrieval evaluation utility |
| NLP | TF-IDF keywords and NMF topics; sentiment/complaint classifier evaluation with existing, reviewed labels, chronological holdout and duplicate-text checks |
| Conversation | Persian/English deterministic fallback or local LLM router; real analysis query; explicit date filters; no generated Python/SQL execution |
| What-if | Explicit price/volume/discount arithmetic; user assumptions; optional cost scenario; no inferred causal elasticity |
| Delivery | React/Vite/TypeScript/Tailwind/Plotly, FastAPI, SQLite/Parquet, notebooks, examples, tests, native and Compose setup |

## Explicit boundaries

This is a single-company implementation. It does not include multi-tenant accounts, staff RBAC, internet-facing deployment, OCR, unattended source-folder ingestion, multiple simultaneous dimension joins, exchange-rate conversion, pretrained/calibrated sentiment models, an automatic complaint taxonomy, validated churn/CLV targets, estimated causal demand elasticity, PDF report generation, or external customer messaging. Numeric classification and regression run only on supplied numeric predictors and existing labels. Text classification uses a separate TF-IDF/logistic pipeline and requires reviewed labels. No XGBoost dependency is added without a measured need.

Native ingestion does not run in a hardened parser sandbox. Size/archive/formula restrictions reduce risk but are not a guarantee that every parser vulnerability or malicious PDF will be caught. Use trusted local input in this deployment. Application-level data encryption and decimal-ledger accounting are separate requirements. Inspect disk/backups permissions for real company data.

Sentiment and complaint classification need domain/language evaluation. Churn labeling needs a defensible outcome horizon and observation window. A synthetic sample label demonstrates the classifier API and must not be interpreted as predictive capability. Missing support is reported explicitly rather than replaced by invented targets or scores.

The forecast minimum of 90 days is an engineering guard, not a statistical guarantee. Model usefulness depends on time span, seasonality, structural changes and backtest results. Its displayed uncertainty band uses pooled absolute residuals and has no guaranteed coverage at each future horizon. RFM clusters are descriptive and their IDs have no intrinsic business meaning.

The model cannot author free-form financial findings. It selects an allowed tool and optional review actions; Python renders computed values. This restricts open-ended explanation but keeps the evidence path inspectable. “Why” questions yield descriptive trend evidence, not a causal conclusion. User-confirmed filters are separate from language interpretation.

## Performance and reproducibility

Workbook sheets are parsed once into Parquet. Analysis uses vectorized operations where appropriate. Changed dataset versions separate all result keys. The process queue is bounded at 16 and defaults to one worker. n8n requests return task receipts promptly. Ollama's model count/concurrency is bounded in Compose. Contexts are bounded, and model calls share one chat model across roles.

The current reader is deliberately bounded and does not implement out-of-core analytics. Larger-than-limit CSV input must be split or handled by a separately designed chunked aggregation flow. Benchmarks report wall/CPU time and completion RSS. A sampling benchmark script records observed peak RSS, not a guaranteed instantaneous OS high-water mark. No performance claim is made without a measurement on the same data and hardware.

Python and npm lock files record the versions used during validation. Image tags are pinned but not digest-locked. Regenerate notebook source cells after any implementation change. The validation record distinguishes unit/integration tests, notebook execution, frontend builds and unperformed live integrations.

## Suggested academic evaluation

Evaluate ingestion correctness using known totals and hostile/ambiguous files. Evaluate mapping against a labeled column-matching set. Evaluate numeric analytics against independent hand calculations. Use chronological forecast backtests and a seasonal baseline. Evaluate retrieval with labeled relevant source IDs. Evaluate question routing and refusal behavior with Persian/English queries, including unsupported claims and prompt injection attempts. Evaluate commercial suggestions through a separately approved business experiment; citations alone do not prove an action will improve sales.


### Code cell · `docs/sources.md`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'docs/sources.md'}"
# Primary references

The implementation was checked against these primary references on 2026-11-03. Runtime validation results are documented separately.

- [n8n Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) describes webhook methods, authentication and test/production registration.
- [n8n HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) describes JSON requests and header credentials.
- [n8n Respond to Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.respondtowebhook/) describes the response-node mode used by the two webhook workflows.
- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) documents shared state, graph nodes and edges.
- [LangChain ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama) documents the local integration package and structured-output support.
- [Ollama embeddings API](https://docs.ollama.com/api/embed) documents local batch embeddings.
- [Ollama local-only configuration](https://docs.ollama.com/faq#how-do-i-disable-ollamas-cloud-features) documents `OLLAMA_NO_CLOUD=1` on the Ollama server.
- [n8n release 2.46.4](https://github.com/n8n-io/n8n/releases/tag/n8n%402.46.4) and [Ollama release 0.40.14](https://github.com/ollama/ollama/releases/tag/v0.40.14) are the image-tag references used in Compose.

These references support the integration design; they do not substitute for importing the workflows into your own n8n instance or evaluating a downloaded local model.


### Code cell · `docs/validation.md`

This is the complete source file. Running the cell writes a separate reconstruction under `notebook_project/`.

In [ ]:
%%writefile "{BUILD_ROOT / 'docs/validation.md'}"
# Validation record

Checked on 2026-11-03 using Python 3.13.16 and Node.js 24.21.0. Package versions are recorded in the Python and npm lockfiles.

| Check | Result |
|---|---|
| Backend unit/integration tests | 42 passed, including upload, approval, background processing and export through FastAPI |
| Financial and preparation guards | Confirmed totals/AOV/profit, gross discounts, per-unit costs, missing values, duplicate reporting, safe joins, Persian numbers and Jalali/native Excel dates |
| ML and agent guards | Chronological backtests, baseline comparisons, existing-label requirements, evidence/source references, refusal of unsupported questions and invented source IDs |
| Local retrieval | FAISS indexing, retrieval attribution and content-hash deduplication exercised with explicit fixture embeddings |
| Text classification | Existing reviewed labels, chronological evaluation and duplicate-text handling exercised on a clearly synthetic fixture |
| Python lint/format | Ruff checks and format checks passed |
| Notebook files | All 11 notebook schemas valid; master code cells executed without errors using a local kernel, with live model/n8n calls disabled |
| React frontend | TypeScript checks and Vite production build passed |
| Workflow artifacts | Three inactive JSON graphs checked for node connections and backend request structure |
| Compose | YAML structure checked; images were not built or started |

The backend test client and Jupyter kernel required execution outside the restricted tool sandbox because it blocks local asynchronous/network primitives. They completed successfully there. The tests use fictional or temporary data and do not contact a company database.

The initial direct npm download timed out. Dependencies were fetched from the official npm registry through a temporary local cache using Python networking. npm verified package integrity, and the delivered lockfile contains official registry URLs. That temporary cache helper is not part of the application.

The production frontend build reports a large Plotly chunk, about 4.84 MB before gzip and 1.47 MB after gzip. It loads dynamically when charts are displayed. The main application JavaScript is about 243 KB before gzip.

## Measured fictional-data workload

For the generated 1,600-row dataset in this session, a single local benchmark observed approximately 1.42 seconds for sample generation/ingestion/mapping/cleaning, 0.43 seconds for the dashboard analyses and 0.52 seconds for forecast training/validation. Sampled peak process RSS was about 327 MB. These are one-run development measurements, not production guarantees. No live LLM latency or n8n execution latency was measured.

## Not verified live

- Importing/publishing the JSON inside a running n8n instance.
- Ollama model inference, embeddings from a real downloaded model, or hardware suitability for a chosen model.
- Docker image builds, service startup or an activated deployment.
- Real-company label quality, forecasting value, retrieval relevance or commercial effectiveness.
- A browser interaction test of the dashboard. The frontend was checked by TypeScript and production build.

The setup guide describes how to perform the n8n/Ollama integration checks. Those integrations are configuration-dependent and have not been presented as completed.


### Run the chapter example

In [ ]:
if "demo_store" not in globals():
    from scripts.notebook_demo import make_demo
    demo_store, demo_settings, demo_id, sample_frame = make_demo(ROOT)
print("Fictional dataset:", demo_id)
from sales_intelligence.reporting import simulate, export
from sales_intelligence.schemas import Simulation
scenario = simulate(demo_store, demo_id, Simulation(price_change=0.05, assumptions_confirmed=True))
assert scenario["causal_prediction"] is False
display(scenario)
for output_format in ["csv", "xlsx", "json", "html"]:
    body, mime, name = export(demo_store, demo_id, output_format)
    assert len(body) > 0
    print(name, mime, len(body), "bytes")
# Run the tests in a terminal, or opt in here. This takes longer than the demonstration cells.
RUN_TESTS = False
if RUN_TESTS:
    import subprocess
    subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, check=True)


### Review checkpoint

Review docs/technical_report.md for delivered capabilities and limits. No local model, n8n server, Docker deployment or user-facing service is started by Run All. Integration cells stay opt-in. Do not use synthetic labels as evidence of predictive business value.

Review this result before moving to the next chapter.